# @devops — Create GitHub Repo & Link Databricks Repo

**Task:** `P0-GITHUB-REPO` (re-opened — repo was never actually created on GitHub)  
**Repo:** `skhage/provable-resilience-intelligence`  
**Credential:** `skhage-github` (gitHubOAuth, ID 358347682222830)  

**Run this notebook on any cluster** — the agent's `executeCode` cannot make external API calls.

Cells:
1. Create GitHub repo via API
2. Configure branch protection (main + develop)
3. Link as Databricks Repo
4. Push initial content from workspace folder

In [0]:
import json
import urllib.request
import urllib.error
import subprocess

# Get GitHub OAuth token via git credential helper (workspace Git credential)
process = subprocess.Popen(
    ["git", "credential", "fill"],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)
stdout, _ = process.communicate(input="protocol=https\nhost=github.com\n\n", timeout=10)
token = None
for line in stdout.strip().split("\n"):
    if line.startswith("password="):
        token = line.split("=", 1)[1]
assert token, "No token found via git credential fill"
print(f"Token obtained: {token[:4]}...{token[-4:]} (len={len(token)})")

gh_headers = {
    "Authorization": f"token {token}",
    "Accept": "application/vnd.github+json",
    "User-Agent": "databricks-devops",
    "X-GitHub-Api-Version": "2022-11-28"
}

# First verify the token works — check authenticated user
try:
    user_req = urllib.request.Request("https://api.github.com/user", headers=gh_headers)
    user_resp = urllib.request.urlopen(user_req, timeout=15)
    user_data = json.loads(user_resp.read().decode())
    scopes = user_resp.headers.get("X-OAuth-Scopes", "none")
    print(f"Authenticated as: {user_data['login']}")
    print(f"Token scopes: {scopes}")
except urllib.error.HTTPError as e:
    body = e.read().decode() if e.fp else ""
    print(f"Auth check failed: {e.code} {e.reason}")
    print(f"Response: {body[:500]}")
    print(f"Headers: {dict(e.headers)}")
    raise

# Check if repo already exists
try:
    check_req = urllib.request.Request(
        "https://api.github.com/repos/skhage/provable-resilience-intelligence",
        headers=gh_headers
    )
    resp = urllib.request.urlopen(check_req, timeout=15)
    repo_data = json.loads(resp.read().decode())
    print(f"\nRepo already exists: {repo_data['html_url']}")
    print(f"Default branch: {repo_data['default_branch']}")
except urllib.error.HTTPError as e:
    if e.code == 404:
        print("\nRepo not found — creating...")
        create_payload = json.dumps({
            "name": "provable-resilience-intelligence",
            "description": "Provable Resilience Intelligence — Governed resilience scoring for Lakelink Fiber enterprise services. DAB-managed Databricks project.",
            "private": False,
            "auto_init": True,
            "has_issues": True,
            "has_projects": True,
            "has_wiki": False
        }).encode()
        create_req = urllib.request.Request(
            "https://api.github.com/user/repos",
            data=create_payload,
            headers={**gh_headers, "Content-Type": "application/json"},
            method="POST"
        )
        create_resp = urllib.request.urlopen(create_req, timeout=30)
        repo = json.loads(create_resp.read().decode())
        print(f"CREATED: {repo['html_url']}")
        print(f"Default branch: {repo['default_branch']}")
        print(f"Clone URL: {repo['clone_url']}")
    else:
        body = e.read().decode() if e.fp else ""
        print(f"Error: {e.code} {e.reason}: {body[:500]}")
        raise

In [0]:
# Create 'develop' branch from main
import urllib.request, urllib.error, json

# Get SHA of main branch HEAD
ref_req = urllib.request.Request(
    "https://api.github.com/repos/skhage/provable-resilience-intelligence/git/ref/heads/main",
    headers=gh_headers
)
ref_resp = urllib.request.urlopen(ref_req, timeout=15)
ref_data = json.loads(ref_resp.read().decode())
main_sha = ref_data["object"]["sha"]
print(f"main HEAD SHA: {main_sha}")

# Create develop branch
try:
    dev_payload = json.dumps({
        "ref": "refs/heads/develop",
        "sha": main_sha
    }).encode()
    dev_req = urllib.request.Request(
        "https://api.github.com/repos/skhage/provable-resilience-intelligence/git/refs",
        data=dev_payload,
        headers=gh_headers,
        method="POST"
    )
    dev_resp = urllib.request.urlopen(dev_req, timeout=15)
    print(f"Created 'develop' branch at {main_sha[:8]}")
except urllib.error.HTTPError as e:
    if e.code == 422:
        print("'develop' branch already exists")
    else:
        raise

In [0]:
# Create a Databricks Repo (Git folder) linked to the GitHub repo
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

try:
    repo = w.repos.create(
        url="https://github.com/skhage/provable-resilience-intelligence.git",
        provider="gitHub",
        path="/Repos/stephen.hage@databricks.com/provable-resilience-intelligence"
    )
    print(f"Databricks Repo created: {repo.path}")
    print(f"Branch: {repo.branch}")
    print(f"URL: {repo.url}")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Databricks Repo already linked.")
    else:
        raise

In [0]:
# This cell copies key files from the workspace folder to the Git repo.
# Run after Steps 1-3 complete successfully.
#
# The Databricks Repo at /Repos/stephen.hage@databricks.com/provable-resilience-intelligence
# is now linked to GitHub. To populate it:
#
# Option A (recommended): Use the Repos UI to commit + push
# Option B: Use the CLI from a local machine:
#   git clone https://github.com/skhage/provable-resilience-intelligence.git
#   cp -r /path/to/workspace/bundle/* .
#   git add -A && git commit -m "[P0-GITHUB-REPO] Initial DAB scaffold"
#   git push origin main
#
# Files to include from the workspace project folder:
# - bundle/           (databricks.yml + resources/*.yml + pipeline/ + dashboards/)
# - .github/          (CI/CD workflows)
# - BRANCH-STRATEGY.md
# - CONTRIBUTING.md
# - README.md
# - .gitignore
# - apps/             (resilience-scorecard app source)
# - pipeline/         (SDP pipeline notebook)
# - tests/

print("GitHub repo and Databricks Repo are ready.")
print("Next: copy project content into the Repo and push to GitHub.")
print("\nRecommended workflow:")
print("  1. Open /Repos/stephen.hage@databricks.com/provable-resilience-intelligence")
print("  2. Copy files from workspace folder")
print("  3. Commit with message: [P0-GITHUB-REPO] Initial DAB scaffold")
print("  4. Push to origin/main")
print("  5. Create PR: main → develop")